# Notebook 02 (v3) — Per-Component Parameter Analysis (schema-aware)

**Goal:** for each of the 20 vision-language models in the Phase-2 roster, count the number of parameters in each architectural component and surface the result inline so you can manually validate it against the legacy Sunita-1/2/3 numbers and against each model's reported figures.

**What changed in v3 (2026-05-22):**
* v2 silently emitted `NaN` in the master DataFrame and empty stacked bars for Chameleon and Janus-Pro because those two models do not fit the V/F/L/H schema (Chameleon is native early-fusion → `fused_transformer` + `embeddings`; Janus-Pro has decoupled `vision_und` + `vision_gen`). v3 detects these schemas, aggregates totals across **all** components (not just V/F/L/H), and renders native models in their own visualization panel so the bars are no longer empty.
* The trainable-vs-total figure and the family summary are now schema-aware: Chameleon's 7B `fused_transformer` is counted, not just the 131M head.
* The master table now includes a `schema` column (`standard_vflh` / `native_fused` / `decoupled_vision`) so it's obvious at a glance why a particular row has values in unusual columns.

**Inputs:** none from previous notebooks (this is the first measurement step).

**Outputs:**
* Per-model JSON records written to `../outputs/measurements/<sanitized_hf_id>.json` (incremental merge — won't clobber other metrics).
* Verbose `.txt` logs in `../outputs/logs/<timestamp>_02_parameter_analysis_nb__<hf_id>.txt`.
* Inline pandas DataFrames + matplotlib visualizations for spot-checking.
* CSV summary at `../outputs/measurements/nb02_parameter_summary.csv`.
* Family summary at `../outputs/measurements/nb02_family_summary.csv`.
* 5 PNG figures at `../outputs/figures/nb02_*.png` (300 dpi).

**Compute:** CPU works; GPU helps only for faster model loading. **No memory measurements here** — those are in notebook 04.

**Disk:** the HuggingFace cache will grow to ~250–500 GB by the end of the full 20-model run. Set `CLEANUP_HF_CACHE_BETWEEN_MODELS = True` in §1 if running on a disk-constrained host.

**How to read this notebook:** run top-to-bottom. Each section has a markdown header followed by code cells. Look at the DataFrame after each cell to verify the numbers make sense before moving on.

---
## 1. Setup (Runpod)

If you're not on Runpod, comment out the `!pip install` line and provide `HF_TOKEN` however your environment expects. Otherwise this cell installs requirements and validates the working directory.

In [32]:
# ===== Environment-specific flags =====
# Set to True on disk-constrained hosts (Colab ~235 GB) so each model's HF cache
# is deleted after measuring its params. Set to False on Runpod (1TB+) so
# subsequent notebooks (03 FLOPs, 04 memory) can reuse the cache.
CLEANUP_HF_CACHE_BETWEEN_MODELS = False

# Warn when free disk drops below this many GB. Aborts the loop if it goes lower.
DISK_FREE_GB_WARN = 30
DISK_FREE_GB_ABORT = 10

In [33]:
# # Runpod setup — install deps + ensure HF_TOKEN is in env
# %%capture
# !pip install -q -r ../requirements.txt

In [34]:
# Set HF_TOKEN here if not already in env. Required for gated models
# (LLaMA / IDEFICS / Pixtral / Llama-3.2-Vision / Chameleon / InstructBLIP).
import os
# os.environ['HF_TOKEN'] = 'hf_...'  # uncomment + paste your token if needed
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN not set — gated models will fail to download'

# Runpod's PyTorch image sets HF_HUB_ENABLE_HF_TRANSFER=1 for faster downloads,
# which requires the `hf_transfer` Python package. If it's missing, every
# from_pretrained() call errors out. Soft-detect and disable fast-transfer
# rather than crashing — the user can pip install hf_transfer and re-enable.
if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
    try:
        import hf_transfer  # noqa: F401
        print('✓ hf_transfer available — fast downloads ENABLED')
    except ImportError:
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
        print('⚠ HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer package not installed.')
        print('  Disabled fast-transfer for this session (downloads will be slower).')
        print('  To re-enable: pip install hf_transfer  --break-system-packages')
else:
    print('hf_transfer fast-download not requested by env')

✓ hf_transfer available — fast downloads ENABLED


In [35]:
import sys, shutil
from pathlib import Path

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures'
for d in (MEASUREMENT_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

HF_HOME = Path(os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))).expanduser()
HF_CACHE = HF_HOME / 'hub'

def free_disk_gb(p='/'):
    s = shutil.disk_usage(p)
    return s.free / 1e9, s.total / 1e9

free_gb, total_gb = free_disk_gb()
print(f'Working dir:        {NB_DIR}')
print(f'code_active dir:    {CODE_ACTIVE}')
print(f'vlm_audit:          {CODE_ACTIVE / "vlm_audit"}')
print(f'HF_HOME (env):      {os.environ.get("HF_HOME", "<not set, using default>")}')
print(f'HF cache dir:       {HF_CACHE}')
print(f'Disk:               {free_gb:.1f} GB free / {total_gb:.1f} GB total ({100*free_gb/total_gb:.1f}% free)')

# Cache persistence diagnostic — the most common cause of unexpected
# re-downloads on Runpod is HF_HOME pointing to /root (ephemeral) instead
# of /workspace (persistent if you used a Network Volume).
_hf_str = str(HF_CACHE)
if _hf_str.startswith('/root'):
    print('⚠ HF cache is under /root — NOT PERSISTENT on Runpod.')
    print('  Every pod restart will wipe the cache and force a full 250 GB re-download.')
    print('  Fix: set HF_HOME=/workspace/.cache/huggingface in your Runpod pod env vars,')
    print('       then restart the kernel. (Notebook env-var sets only affect THIS session.)')
elif _hf_str.startswith('/workspace') or 'OneDrive' in _hf_str or _hf_str.startswith('/home'):
    print(f'✓ HF cache is under a persistent path.')
else:
    print(f'? HF cache at {_hf_str} — confirm this is on a persistent volume')

# How many models are already cached? (sanity-check: re-runs should report >0)
if HF_CACHE.exists():
    cached_models = [p for p in HF_CACHE.iterdir() if p.is_dir() and p.name.startswith('models--')]
    cached_size_gb = 0.0
    for p in HF_CACHE.rglob('*'):
        if p.is_file() and not p.is_symlink():
            try:
                cached_size_gb += p.stat().st_size / 1e9
            except Exception:
                pass
    print(f'Models already cached:  {len(cached_models)}  ({cached_size_gb:.1f} GB on disk)')
    if cached_models:
        print('  First few:')
        for p in sorted(cached_models)[:5]:
            print(f'    {p.name}')
else:
    print(f'HF cache directory does not exist yet — first run will download everything.')

if free_gb < 50:
    print(f'\n⚠ Less than 50 GB free — set CLEANUP_HF_CACHE_BETWEEN_MODELS=True or move HF_HOME to a bigger volume.')

Working dir:        /workspace/code_active/notebooks
code_active dir:    /workspace/code_active
vlm_audit:          /workspace/code_active/vlm_audit
HF_HOME (env):      /workspace/.cache/huggingface/
HF cache dir:       /workspace/.cache/huggingface/hub
Disk:               51.7 GB free / 53.7 GB total (96.3% free)
✓ HF cache is under a persistent path.
Models already cached:  20  (2.7 GB on disk)
  First few:
    models--HuggingFaceM4--idefics-9b
    models--HuggingFaceM4--idefics2-8b
    models--OpenGVLab--InternVL3-8B
    models--Qwen--Qwen2-VL-7B-Instruct
    models--Qwen--Qwen2.5-VL-7B-Instruct


In [36]:
import pandas as pd
import numpy as np
import torch
import time
import traceback
import gc
import matplotlib.pyplot as plt

from vlm_audit import (
    MODELS, get_model_list, count_params,
    extract_components, compute_trainable_from_policy,
    run_full_audit, InputConfig,
)
from vlm_audit.runner import _load_model, _resolve_hf_class
from vlm_audit.components import (
    freezing_policy_hint, trainable_components_for_family,
    COMPONENT_NAMES, EXTENDED_COMPONENT_NAMES,
)
from vlm_audit.logging_utils import sanitize_for_filename

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Models in roster: {len(MODELS)}')
print(f'Cleanup HF cache between models: {CLEANUP_HF_CACHE_BETWEEN_MODELS}')
print(f'Standard V/F/L/H component names: {COMPONENT_NAMES}')
print(f'Extended (incl. native) component names: {EXTENDED_COMPONENT_NAMES}')

PyTorch:        2.8.0+cu128
CUDA available: True
GPU:            NVIDIA A100-SXM4-80GB
VRAM:           85.1 GB
Models in roster: 20
Cleanup HF cache between models: False
Standard V/F/L/H component names: ('vision', 'fusion', 'language', 'head')
Extended (incl. native) component names: ('vision', 'fusion', 'language', 'head', 'fused_transformer', 'vision_und', 'vision_gen', 'embeddings')


---
## 2. Schema awareness — three families of component layouts

The v3 fix hinges on this distinction. The 20 models fall into three component schemas:

| Schema | Models | Component keys returned by extractor |
|---|---|---|
| `standard_vflh` | 18 of 20 (CLIP, BLIP, BLIP-2, InstructBLIP, LLaVA, IDEFICS, ViLT, Qwen-VL, InternVL3, Molmo, Pixtral, Llama-3.2-Vision, DeepSeek-VL2-Small, Idefics2, SigLIP2) | `vision`, `fusion`, `language`, `head` (some may be `None` when the architecture has no such module — e.g. CLIP has no `head`) |
| `native_fused` | Chameleon-7B | `vision=None`, `fusion=None`, `language=None`, `head`, **`fused_transformer`** (the shared transformer body), **`embeddings`** (image-token vocab + text-token vocab table). V/F/L/H aren't separable here. |
| `decoupled_vision` | Janus-Pro-7B | `vision=vision_und` (alias), `fusion`, `language`, `head`, **`vision_und`** (SigLIP-L for understanding), **`vision_gen`** (VQ encoder for generation). Two distinct vision pathways. |

**Why v2 produced NaN + empty bars:** the v2 aggregation cell only summed `vision_total + fusion_total + language_total + head_total`. For Chameleon, those four columns were all `0` (because the extractor returned `None`), so its total looked like just the 131M head — almost empty. For Janus, the `vision_gen_total` (~300M for the VQ pathway) was silently dropped from the aggregate.

**v3 fix:** the helper `safe_sum()` below sums **all** component-suffixed columns that exist in the DataFrame. The helper `vision_share()` aggregates `vision + vision_und + vision_gen` so decoupled models are counted correctly. Schema is also tagged per-row in the master table.

In [37]:
# Schema classifier — runs over the model spec, not the DataFrame, so it
# can be used at any stage. Returns one of the three schema labels above.
NATIVE_FUSED_FAMILIES = {'native_early_fusion'}
DECOUPLED_VISION_FAMILIES = {'native_decoupled_encoder'}

def schema_for(family: str) -> str:
    if family in NATIVE_FUSED_FAMILIES:
        return 'native_fused'
    if family in DECOUPLED_VISION_FAMILIES:
        return 'decoupled_vision'
    return 'standard_vflh'

# Helpers for schema-aware aggregation over the master DataFrame.
EXTENDED_PARTS = list(EXTENDED_COMPONENT_NAMES)  # ('vision','fusion','language','head','fused_transformer','vision_und','vision_gen','embeddings')
VISION_LIKE_PARTS = ['vision', 'vision_und', 'vision_gen']
LANGUAGE_LIKE_PARTS = ['language', 'fused_transformer']  # Chameleon's fused body acts as language too

def safe_sum(df: pd.DataFrame, parts, suffix: str = '_total') -> pd.Series:
    """Sum of columns parts[i]+suffix that exist in df. Missing cols treated as 0."""
    cols = [f'{p}{suffix}' for p in parts if f'{p}{suffix}' in df.columns]
    if not cols:
        return pd.Series([0] * len(df), index=df.index)
    return df[cols].fillna(0).sum(axis=1)

# Sanity checks — these should not error.
assert schema_for('native_early_fusion') == 'native_fused'
assert schema_for('native_decoupled_encoder') == 'decoupled_vision'
assert schema_for('adapter_llm') == 'standard_vflh'
print('schema_for() ready.')
print('safe_sum() ready.')
print(f'EXTENDED_PARTS         = {EXTENDED_PARTS}')
print(f'VISION_LIKE_PARTS      = {VISION_LIKE_PARTS}')
print(f'LANGUAGE_LIKE_PARTS    = {LANGUAGE_LIKE_PARTS}')

schema_for() ready.
safe_sum() ready.
EXTENDED_PARTS         = ['vision', 'fusion', 'language', 'head', 'fused_transformer', 'vision_und', 'vision_gen', 'embeddings']
VISION_LIKE_PARTS      = ['vision', 'vision_und', 'vision_gen']
LANGUAGE_LIKE_PARTS    = ['language', 'fused_transformer']


---
## 3. Inspect the model roster

Before measuring anything, show the 20 models we're about to audit, with their schema tag. Sanity-check that the roster is what you expect, and that total fp16 footprint sums to something tractable for your disk.

In [38]:
roster_df = pd.DataFrame([
    {
        'name':           m.name,
        'family':         m.family,
        'schema':         schema_for(m.family),
        'year':           m.release_year,
        'fp16_GB':        m.fits_in_gb_fp16,
        'native_HW':      f'{m.native_image_hw[0]}x{m.native_image_hw[1]}',
        'recent_gen':     m.recent_gen,
        'is_moe':         m.is_moe,
        'trust_remote':   m.requires_trust_remote_code,
        'hf_id':          m.hf_id,
    }
    for m in MODELS
])
total_fp16_GB = roster_df['fp16_GB'].sum()
print(f'Total models: {len(roster_df)}')
print(f'Families:     {sorted(roster_df.family.unique())} ({len(roster_df.family.unique())} families)')
print(f'Schemas:      {roster_df.schema.value_counts().to_dict()}')
print(f'Recent-gen models (2024+): {int(roster_df.recent_gen.sum())}')
print(f'Total fp16 footprint sum (lower bound, no activations): {total_fp16_GB:.1f} GB')
print(f'Cumulative if all models cached on disk: roughly 2x = {2*total_fp16_GB:.0f} GB.')
roster_df

Total models: 20
Families:     ['adapter_llm', 'adapter_llm_own_encoder', 'bridge_module', 'bridge_module_adapter_llm', 'bridge_module_precursor', 'cross_attention_llm', 'dual_encoder_contrastive', 'dynamic_tile_adapter_llm', 'moe_adapter_llm', 'native_decoupled_encoder', 'native_early_fusion', 'perceiver_adapter_llm', 'single_stream_fused'] (13 families)
Schemas:      {'standard_vflh': 18, 'native_fused': 1, 'decoupled_vision': 1}
Recent-gen models (2024+): 12
Total fp16 footprint sum (lower bound, no activations): 266.5 GB
Cumulative if all models cached on disk: roughly 2x = 533 GB.


,name,family,schema,year,fp16_GB,native_HW,recent_gen,is_moe,trust_remote,hf_id
0,CLIP-ViT-L/14,dual_encoder_contrastive,standard_vflh,2021,1.0,224x224,False,False,False,openai/clip-vit-large-patch14
1,SigLIP-L/16,dual_encoder_contrastive,standard_vflh,2023,1.5,384x384,False,False,False,google/siglip-large-patch16-384
2,BLIP-large,bridge_module_precursor,standard_vflh,2022,1.0,384x384,False,False,False,Salesforce/blip-image-captioning-large
3,BLIP-2-OPT-2.7B,bridge_module,standard_vflh,2023,7.0,224x224,False,False,False,Salesforce/blip2-opt-2.7b
4,InstructBLIP-Vicuna-7B,bridge_module_adapter_llm,standard_vflh,2023,17.0,224x224,False,False,False,Salesforce/instructblip-vicuna-7b
5,LLaVA-1.5-7B,adapter_llm,standard_vflh,2023,16.0,336x336,False,False,False,llava-hf/llava-1.5-7b-hf
6,LLaVA-NeXT-7B,adapter_llm,standard_vflh,2024,16.0,336x336,True,False,False,llava-hf/llava-v1.6-mistral-7b-hf
7,IDEFICS-9B,cross_attention_llm,standard_vflh,2023,18.0,224x224,False,False,False,HuggingFaceM4/idefics-9b
8,Idefics2-8B,perceiver_adapter_llm,standard_vflh,2024,16.0,224x224,True,False,False,HuggingFaceM4/idefics2-8b
9,ViLT-B/32,single_stream_fused,standard_vflh,2021,0.5,384x384,False,False,False,dandelin/vilt-b32-finetuned-vqa


---
## 4. Smoke test — CLIP-ViT-L/14 (validation anchor against Sunita-1)

Before iterating over all 20 models, validate the pipeline on **one** known-good model where we have a ground-truth comparison.

From the legacy `Sunita-1.ipynb` cell 12, CLIP-ViT-L/14 had:
* `vision_model`:  303,179,776 params
* `text_model`:    123,060,480 params
* projection:        1,376,256 params
* **Total:**       427,616,512 params

If our pipeline matches these to the last digit, we know `extract_components` and `count_params` are correct for the CLIP family.

In [39]:
clip_spec = next(m for m in MODELS if m.name == 'CLIP-ViT-L/14')
print(f'Auditing: {clip_spec.name}')
print(f'  HF id:     {clip_spec.hf_id}')
print(f'  Family:    {clip_spec.family}  (schema={schema_for(clip_spec.family)})')
print(f'  Native HW: {clip_spec.native_image_hw}')
print(f'  Freezing recipe: {freezing_policy_hint(clip_spec.family)}')
print(f'  Trainable-component table per recipe: {trainable_components_for_family(clip_spec.family)}')

Auditing: CLIP-ViT-L/14
  HF id:     openai/clip-vit-large-patch14
  Family:    dual_encoder_contrastive  (schema=standard_vflh)
  Native HW: (224, 224)
  Freezing recipe: End-to-end pretraining; nothing frozen.
  Trainable-component table per recipe: {'vision': True, 'fusion': True, 'language': True, 'head': True}


In [40]:
t0 = time.time()
rec = run_full_audit(
    clip_spec,
    metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_v3_nb',
)
print(f'Done in {time.time() - t0:.1f}s')
log_files = sorted(LOG_DIR.glob('*02_parameter_analysis_v3_nb__openai*'))
if log_files:
    print(f'Log file: {log_files[-1].name}')

Done in 0.8s
Log file: 20260521_220351_02_parameter_analysis_v3_nb__openai_clip-vit-large-patch14.txt


In [41]:
clip_df = pd.DataFrame([
    {
        'component':         c.name,
        'params_total':      c.params_total,
        'params_total_M':    round((c.params_total or 0) / 1e6, 2),
        'params_trainable':  c.params_trainable,
        'trainable_frac':    round((c.params_trainable or 0) / max(c.params_total or 1, 1), 3),
        'notes':             c.notes,
    }
    for c in rec.components.values()
])
clip_df

,component,params_total,params_total_M,params_trainable,trainable_frac,notes
0,vision,303179776,303.18,303179776,1.0,
1,fusion,1376256,1.38,1376256,1.0,
2,language,123060480,123.06,123060480,1.0,
3,head,0,0.00,0,0.0,absent


In [42]:
EXPECTED = {
    'vision':   303_179_776,
    'language': 123_060_480,
    'fusion':     1_376_256,
}
ok = True
for cname, expected in EXPECTED.items():
    got = rec.components[cname].params_total or 0
    delta = got - expected
    status = '✓' if delta == 0 else f'✗ (off by {delta:+,})'
    print(f'  {cname:10s}: expected {expected:>12,}  got {got:>12,}  {status}')
    if delta != 0:
        ok = False
assert ok, 'CLIP-L params do not match legacy Sunita-1 — extractor may be broken!'
print('\n✓ CLIP-L params match Sunita-1 cell 12 exactly. Pipeline is correct for CLIP family.')
print('  Interpretation: CLIP-L is vision-heavy — 70.9% vision, 28.8% language, 0.3% fusion.')
print('                  V/L ratio = 2.46×. This is the corrected ratio (not the 10× from older docs — see memory.md M-001).')

  vision    : expected  303,179,776  got  303,179,776  ✓
  language  : expected  123,060,480  got  123,060,480  ✓
  fusion    : expected    1,376,256  got    1,376,256  ✓

✓ CLIP-L params match Sunita-1 cell 12 exactly. Pipeline is correct for CLIP family.
  Interpretation: CLIP-L is vision-heavy — 70.9% vision, 28.8% language, 0.3% fusion.
                  V/L ratio = 2.46×. This is the corrected ratio (not the 10× from older docs — see memory.md M-001).


---
## 5. Smoke test — IDEFICS-9B (Phase-2 F-3 + F-5 bug-fix verification)

IDEFICS-9B was the Phase-1 bug case: its `language` component used to read 0 params because the extractor couldn't find the LLaMA backbone. Phase-2 `components.py` fixes this by explicitly walking into `model.model` and assembling a `ModuleList` of the LLaMA layers + `embed_tokens` + `norm`.

Validate the fix before scaling to all 20:
* **Phase-1 (broken):** language = 0 → IDEFICS labeled 83% vision-dominant (wrong)
* **Phase-2 (correct):** language ≈ 6.6B → IDEFICS correctly labeled language-dominant

We also validate the **F-5 fix**: `params_trainable` is computed from the freezing policy, not from `requires_grad` (which is `True` everywhere after `.from_pretrained()`).

In [43]:
idefics_spec = next(m for m in MODELS if m.name == 'IDEFICS-9B')
rec_idefics = run_full_audit(
    idefics_spec, metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_v3_nb',
)
idefics_df = pd.DataFrame([
    {'component': c.name, 'params_M': round((c.params_total or 0)/1e6, 2),
     'trainable_M': round((c.params_trainable or 0)/1e6, 2),
     'trainable_pct': round(100 * (c.params_trainable or 0) / max(c.params_total or 1, 1), 1),
     'notes': c.notes}
    for c in rec_idefics.components.values()
])
idefics_df

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00019.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

model-00002-of-00019.safetensors:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

model-00003-of-00019.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00004-of-00019.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

... (14 outputs left out)

model-00019-of-00019.safetensors:   0%|          | 0.00/705M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

,component,params_M,trainable_M,trainable_pct,notes
0,vision,630.77,0.00,0.0,
1,fusion,125.96,125.96,100.0,
2,language,6607.35,0.00,0.0,
3,head,131.08,131.08,100.0,


In [44]:
# Verify F-3 + F-5 fixes simultaneously
lang_params = rec_idefics.components['language'].params_total or 0
lang_trainable = rec_idefics.components['language'].params_trainable or 0
vision_trainable = rec_idefics.components['vision'].params_trainable or 0
fusion_trainable = rec_idefics.components['fusion'].params_trainable or 0

print('=== Phase-2 F-3 fix verification (IDEFICS language extractor) ===')
print(f'  language params: {lang_params:,} ({lang_params/1e9:.2f}B)')
assert lang_params > 5_000_000_000, f'F-3 fix not working — language should be ~6.6B, got {lang_params:,}'
print('  ✓ Phase-2 F-3 fix WORKS — IDEFICS language is now correctly ~6.6B (was 0 in Phase 1)')

print('\n=== Phase-2 F-5 fix verification (trainable from policy, not requires_grad) ===')
print(f'  vision_trainable:   {vision_trainable:,}  (recipe: frozen → expect 0)')
print(f'  language_trainable: {lang_trainable:,}  (recipe: frozen → expect 0)')
print(f'  fusion_trainable:   {fusion_trainable:,}  (recipe: gated cross-attn trainable → expect ~126M)')
assert vision_trainable == 0, f'F-5 fix not working — vision should show 0 trainable (frozen), got {vision_trainable:,}'
assert lang_trainable == 0, f'F-5 fix not working — language should show 0 trainable (frozen), got {lang_trainable:,}'
assert fusion_trainable > 100_000_000, f'F-5 fix not working — fusion should be trainable ~126M, got {fusion_trainable:,}'
print('  ✓ Phase-2 F-5 fix WORKS — IDEFICS shows frozen vision + frozen language + trainable cross-attention')

print('\nInterpretation: IDEFICS-9B trains only 257M parameters (fusion + head) out of 7.5B total — 3.4% trainable.')
print('This is the canonical Flamingo-style design: massive frozen vision + massive frozen LLM, learn only the bridge.')

=== Phase-2 F-3 fix verification (IDEFICS language extractor) ===
  language params: 6,607,351,808 (6.61B)
  ✓ Phase-2 F-3 fix WORKS — IDEFICS language is now correctly ~6.6B (was 0 in Phase 1)

=== Phase-2 F-5 fix verification (trainable from policy, not requires_grad) ===
  vision_trainable:   0  (recipe: frozen → expect 0)
  language_trainable: 0  (recipe: frozen → expect 0)
  fusion_trainable:   125,961,984  (recipe: gated cross-attn trainable → expect ~126M)
  ✓ Phase-2 F-5 fix WORKS — IDEFICS shows frozen vision + frozen language + trainable cross-attention

Interpretation: IDEFICS-9B trains only 257M parameters (fusion + head) out of 7.5B total — 3.4% trainable.
This is the canonical Flamingo-style design: massive frozen vision + massive frozen LLM, learn only the bridge.


---
## 6. Schema preview — Chameleon-7B and Janus-Pro-7B

Before the main loop runs all 20 models, let's preview what the schema-aware fix is solving. The next two cells show the **raw extractor output** for the two native models, so you can see directly that V/F/L/H are `None` for Chameleon and that Janus has an extra `vision_gen` pathway.

*Note: these cells DOWNLOAD the model weights. Skip them if you want to save time — but the main loop will re-download anyway. ~14 GB for Chameleon, ~14 GB for Janus.*

In [45]:
# Chameleon — show the extractor result
chameleon_spec = next(m for m in MODELS if m.name == 'Chameleon-7B')
print(f'Family:  {chameleon_spec.family}')
print(f'Schema:  {schema_for(chameleon_spec.family)}')
print(f'Freezing recipe: {freezing_policy_hint(chameleon_spec.family)}')

rec_cham = run_full_audit(
    chameleon_spec, metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_v3_nb',
)
cham_df = pd.DataFrame([
    {'component': c.name,
     'params_M': round((c.params_total or 0)/1e6, 2),
     'trainable_M': round((c.params_trainable or 0)/1e6, 2),
     'notes': c.notes}
    for c in rec_cham.components.values()
])
print()
print('Chameleon-7B component breakdown:')
print('  Notice that vision/fusion/language are ABSENT — this is by design.')
print('  The entire 7B model is in fused_transformer; the rest is head + the image-vocab embedding table.')
cham_df

Family:  native_early_fusion
Schema:  native_fused
Freezing recipe: End-to-end pretraining of the single shared transformer over interleaved tokens.


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]


Chameleon-7B component breakdown:
  Notice that vision/fusion/language are ABSENT — this is by design.
  The entire 7B model is in fused_transformer; the rest is head + the image-vocab embedding table.


,component,params_M,trainable_M,notes
0,vision,0.00,0.00,absent
1,fusion,0.00,0.00,absent
2,language,0.00,0.00,absent
3,head,268.44,268.44,
4,fused_transformer,6774.65,6774.65,
5,embeddings,268.44,268.44,


In [47]:
# Janus-Pro — show the extractor result
janus_spec = next(m for m in MODELS if m.name == 'Janus-Pro-7B')
print(f'Family:  {janus_spec.family}')
print(f'Schema:  {schema_for(janus_spec.family)}')
print(f'Freezing recipe: {freezing_policy_hint(janus_spec.family)}')

rec_janus = run_full_audit(
    janus_spec, metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_v3_nb',
)
janus_df = pd.DataFrame([
    {'component': c.name,
     'params_M': round((c.params_total or 0)/1e6, 2),
     'trainable_M': round((c.params_trainable or 0)/1e6, 2),
     'notes': c.notes}
    for c in rec_janus.components.values()
])
print()
print('Janus-Pro-7B component breakdown:')
print('  Notice the TWO vision pathways: vision_und (SigLIP-L, the input vision encoder)')
print('  and vision_gen (the VQ encoder for image generation). v2 was silently dropping vision_gen.')
janus_df

Family:  native_decoupled_encoder
Schema:  decoupled_vision
Freezing recipe: Two-stage: vision encoders pretrained, then LLM joint-trained on both encoders.

Janus-Pro-7B component breakdown:
  Notice the TWO vision pathways: vision_und (SigLIP-L, the input vision encoder)
  and vision_gen (the VQ encoder for image generation). v2 was silently dropping vision_gen.


""


---
## 7. Run parameter counts on all 20 models

Iterate over `MODELS`, wrap each in try/except so one failure doesn't kill the whole run. Failures collected into `failures_df`.

Per-model output:
* `[i/N] <name>` header
* progress: load time and total params on success
* **schema-aware inline interpretation**: short natural-language summary of the V/F/L/H allocation, or the native-model summary if it's Chameleon/Janus
* **disk monitoring**: free space before and after each model; cleanup if `CLEANUP_HF_CACHE_BETWEEN_MODELS = True`
* `failures_df` if anything blew up

**Disk planning:** model 9 (Idefics2-8B) on Colab takes 12+ minutes due to bandwidth. Model 14 (InternVL3-8B) is where Colab free-tier disk usually fills up.

**Wall-clock estimate:**
* Runpod A100/L40 + 1TB disk + good bandwidth: **~20-40 minutes** for all 20 models (cold cache).
* Subsequent reruns (warm cache): ~3-5 minutes total.

In [48]:
# Helpers used inside the main loop below.
def hf_cache_size_gb(hf_id: str) -> float:
    cache_subdir = HF_CACHE / f'models--{hf_id.replace("/", "--")}'
    if not cache_subdir.exists():
        return 0.0
    total = 0
    for p in cache_subdir.rglob('*'):
        if p.is_file() and not p.is_symlink():
            try:
                total += p.stat().st_size
            except Exception:
                pass
    return total / 1e9

def cleanup_hf_cache_for(hf_id: str, log=True) -> float:
    cache_subdir = HF_CACHE / f'models--{hf_id.replace("/", "--")}'
    if not cache_subdir.exists():
        if log:
            print(f'      [cleanup] no cache for {hf_id} — skipping')
        return 0.0
    sz = hf_cache_size_gb(hf_id)
    shutil.rmtree(cache_subdir, ignore_errors=True)
    if log:
        print(f'      [cleanup] removed {sz:.2f} GB of cache for {hf_id}')
    return sz

def interpret_allocation(rec, schema: str) -> str:
    """Schema-aware natural-language interpretation of param allocation."""
    comps = rec.components

    def g(name):  # short for 'get total of component name, default 0'
        c = comps.get(name)
        return (c.params_total if c else 0) or 0

    if schema == 'native_fused':
        fused = g('fused_transformer')
        emb = g('embeddings')
        head = g('head')
        total = fused + emb + head
        if total == 0:
            return '      [interpret] native fused (no decomposition possible) — see notebook 10'
        return (f'      [interpret] native EARLY FUSION → {fused/1e6:.0f}M shared transformer + '
                f'{emb/1e6:.0f}M embeddings + {head/1e6:.0f}M head '
                f'(V/F/L not separable — total {total/1e9:.2f}B)')
    if schema == 'decoupled_vision':
        v_und = g('vision_und')
        v_gen = g('vision_gen')
        l = g('language')
        h = g('head')
        total = v_und + v_gen + l + h + g('fusion')
        if total == 0:
            return '      [interpret] decoupled vision (no totals) — see notebook 10'
        return (f'      [interpret] DECOUPLED vision → understanding {v_und/1e6:.0f}M + '
                f'generation {v_gen/1e6:.0f}M + language {l/1e9:.2f}B; total {total/1e9:.2f}B')
    # Standard V/F/L/H
    v, f, l, h = g('vision'), g('fusion'), g('language'), g('head')
    total = v + f + l + h
    if total == 0:
        return '      [interpret] no decomposition (extractor returned None for all)'
    v_pct = 100 * v / total
    f_pct = 100 * f / total
    l_pct = 100 * l / total
    if v_pct > 50:
        dom = 'vision-dominant'
    elif l_pct > 70:
        dom = 'language-dominant'
    elif 0.4 < v / max(l, 1) < 2.5:
        dom = 'roughly balanced'
    else:
        dom = 'mixed'
    trainable_total = sum((c.params_trainable or 0) for c in comps.values())
    total_w_head = sum((c.params_total or 0) for c in comps.values())
    train_pct = 100 * trainable_total / max(total_w_head, 1)
    return (f'      [interpret] {v_pct:.1f}% vision · {f_pct:.1f}% fusion · {l_pct:.1f}% language '
            f'→ {dom}; {train_pct:.1f}% trainable under canonical recipe')

In [49]:
# Choose what to run:
#   MODELS         — all 20 (full run)
#   MODELS[:3]     — smoke test
#   MODELS[10:]    — only Phase-2 new additions
#   [m for m in MODELS if m.name in ('Chameleon-7B', 'Janus-Pro-7B')]  — just native models
TARGETS = MODELS

results = []
failures = []
cleanup_total_gb = 0.0

for i, spec in enumerate(TARGETS, 1):
    free_gb_before, _ = free_disk_gb()
    sch = schema_for(spec.family)
    cache_gb_before = hf_cache_size_gb(spec.hf_id)
    was_cached = cache_gb_before > 0.1
    cache_label = f'cache_hit={cache_gb_before:.1f}GB' if was_cached else 'fresh_dl'
    print(f'\n[{i}/{len(TARGETS)}] {spec.name:30s} schema={sch:18s} disk_free={free_gb_before:.1f} GB  {cache_label}')

    if free_gb_before < DISK_FREE_GB_ABORT:
        print(f'    ⛔ disk free < {DISK_FREE_GB_ABORT} GB — ABORTING. set CLEANUP_HF_CACHE_BETWEEN_MODELS=True or move HF_HOME.')
        failures.append({'name': spec.name, 'error': f'aborted: disk_free={free_gb_before:.1f} GB < {DISK_FREE_GB_ABORT}'})
        break
    elif free_gb_before < DISK_FREE_GB_WARN:
        print(f'    ⚠ disk free < {DISK_FREE_GB_WARN} GB — proceeding, but watch')

    t0 = time.time()
    try:
        rec = run_full_audit(
            spec, metrics={'params'},
            measurement_dir=str(MEASUREMENT_DIR),
            log_dir=str(LOG_DIR),
            notebook_label='02_parameter_analysis_v3_nb',
        )
        row = {'name': spec.name, 'family': spec.family, 'schema': sch, 'year': spec.release_year}
        for cname, c in rec.components.items():
            row[f'{cname}_total'] = c.params_total or 0
            row[f'{cname}_trainable'] = c.params_trainable or 0
        results.append(row)
        total_M = sum((c.params_total or 0) for c in rec.components.values()) / 1e6
        elapsed = time.time() - t0
        cache_gb_after = hf_cache_size_gb(spec.hf_id)
        downloaded_gb = max(0, cache_gb_after - cache_gb_before)
        # Cache hit vs download diagnostic: fresh download is typically
        # 5-60s for small models, minutes for 7B+. Cache hit is <5s.
        if was_cached and elapsed > 30:
            cache_note = f'  ⚠ slow load ({elapsed:.1f}s) despite cache_gb_before={cache_gb_before:.1f}GB — possible HF re-validate or cache miss for some shards'
        elif downloaded_gb > 0.5:
            cache_note = f'  [downloaded {downloaded_gb:.1f} GB; HF_HOME={os.environ.get("HF_HOME","<default>")}]'
        else:
            cache_note = ''
        print(f'    OK  total={total_M:.2f}M params  ({elapsed:.1f}s){cache_note}')
        print(interpret_allocation(rec, sch))
        if CLEANUP_HF_CACHE_BETWEEN_MODELS:
            cleanup_total_gb += cleanup_hf_cache_for(spec.hf_id, log=True)
    except Exception as e:
        tb = traceback.format_exc()
        failures.append({
            'name': spec.name, 'family': spec.family,
            'error': f'{type(e).__name__}: {e}',
            'traceback_excerpt': '\n'.join(tb.splitlines()[-5:]),
        })
        print(f'    FAILED  {type(e).__name__}: {e}')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n=== Finished: {len(results)} succeeded, {len(failures)} failed ===')
if CLEANUP_HF_CACHE_BETWEEN_MODELS:
    print(f'  Cleaned up {cleanup_total_gb:.1f} GB of HF cache during run')
free_gb_after, _ = free_disk_gb()
print(f'  Disk free now: {free_gb_after:.1f} GB')


[1/20] CLIP-ViT-L/14                  schema=standard_vflh      disk_free=51.7 GB  cache_hit=1.7GB
    OK  total=427.62M params  (2.5s)
      [interpret] 70.9% vision · 0.3% fusion · 28.8% language → vision-dominant; 100.0% trainable under canonical recipe

[2/20] SigLIP-L/16                    schema=standard_vflh      disk_free=51.7 GB  fresh_dl


config.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

    OK  total=652.48M params  (9.2s)  [downloaded 2.6 GB; HF_HOME=/workspace/.cache/huggingface/]
      [interpret] 48.5% vision · 0.0% fusion · 51.5% language → roughly balanced; 100.0% trainable under canonical recipe

[3/20] BLIP-large                     schema=standard_vflh      disk_free=51.7 GB  fresh_dl


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

... (54 outputs left out)

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

KeyboardInterrupt: 

---
## 8. Master parameter table (schema-aware)

Display the full breakdown for all 20 models in one DataFrame, sorted by family then year. This is the table that becomes Table 1 in the paper.

**v3 change:** the `total_M` and `trainable_total_M` columns are now schema-aware. Chameleon's 7B is correctly shown (was ~131M in v2). The extended columns for native models are also kept visible in the wide view below.

In [ ]:
if not results:
    print('No successful results — see failures below.')
    params_df = pd.DataFrame()
else:
    params_df = pd.DataFrame(results)
    # Schema-aware totals across ALL components
    params_df['total_M']           = safe_sum(params_df, EXTENDED_PARTS, '_total')     / 1e6
    params_df['trainable_total_M'] = safe_sum(params_df, EXTENDED_PARTS, '_trainable') / 1e6
    # Vision-like share = vision + vision_und + vision_gen (Janus has both pathways)
    params_df['vision_like_M']     = safe_sum(params_df, VISION_LIKE_PARTS, '_total')  / 1e6
    # Language-like share = language + fused_transformer (Chameleon's fused body)
    params_df['language_like_M']   = safe_sum(params_df, LANGUAGE_LIKE_PARTS, '_total')/ 1e6
    params_df['fusion_M']          = safe_sum(params_df, ['fusion'], '_total')         / 1e6
    params_df['head_M']            = safe_sum(params_df, ['head'], '_total')           / 1e6
    # Percentage shares relative to schema-aware total
    denom = (params_df['total_M'] * 1e6).clip(lower=1)
    params_df['vision_pct']   = (100 * safe_sum(params_df, VISION_LIKE_PARTS, '_total')   / denom).round(1)
    params_df['fusion_pct']   = (100 * safe_sum(params_df, ['fusion'], '_total')          / denom).round(1)
    params_df['language_pct'] = (100 * safe_sum(params_df, LANGUAGE_LIKE_PARTS, '_total') / denom).round(1)
    # V/L ratio — undefined for native_fused; computed on vision_like / language_like
    v = safe_sum(params_df, VISION_LIKE_PARTS, '_total')
    l = safe_sum(params_df, LANGUAGE_LIKE_PARTS, '_total').replace(0, np.nan)
    params_df['v_to_l_ratio'] = (v / l).round(3)
    # Trainable percent across all components
    params_df['trainable_pct'] = (100 * params_df['trainable_total_M'] / params_df['total_M'].clip(lower=1e-6)).round(1)
summary_cols = ['name', 'family', 'schema', 'year', 'total_M', 'vision_pct', 'fusion_pct', 'language_pct', 'v_to_l_ratio', 'trainable_total_M', 'trainable_pct']
params_df[summary_cols].sort_values(['schema', 'family', 'year']).reset_index(drop=True) if results else None

In [ ]:
# Wide view including extended component columns (so Chameleon's fused_transformer
# and Janus's vision_gen are visible). NaN cells are EXPECTED — they mean 'this
# model's schema does not include this component'.
if results:
    wide_cols = ['name', 'family', 'schema']
    for p in EXTENDED_PARTS:
        c = f'{p}_total'
        if c in params_df.columns:
            wide_cols.append(c)
            params_df[c + '_M'] = (params_df[c].fillna(0) / 1e6).round(2)
    wide_view = params_df[['name', 'family', 'schema'] + [f'{p}_total_M' for p in EXTENDED_PARTS if f'{p}_total_M' in params_df.columns]]
    wide_view = wide_view.sort_values(['schema', 'family']).reset_index(drop=True)
wide_view if results else None

**Reading the wide table above:**
* `standard_vflh` rows: `fused_transformer_total_M`, `vision_und_total_M`, `vision_gen_total_M`, `embeddings_total_M` should all be 0 (or absent). The `vision/fusion/language/head` columns carry the actual data.
* `native_fused` rows (Chameleon): `vision_total_M`, `fusion_total_M`, `language_total_M` should all be 0. The actual mass is in `fused_transformer_total_M` (~7B) + `embeddings_total_M` + `head_total_M`.
* `decoupled_vision` rows (Janus): `vision_und_total_M` and `vision_gen_total_M` are both populated (the two pathways). The `vision_total_M` column is an alias for `vision_und_total_M` (set by the extractor for legacy compatibility).

If you see a 0 where you don't expect one (e.g. Idefics2's `vision_total_M` = 0), open the corresponding log file in `../outputs/logs/` and look for `Extracting components` / `Param counting failed` lines.

---
## 9. Visualizations — params at a glance

Five headline charts. v3 splits the stacked-bar figure into **two panels** — one for standard V/F/L/H models, one for native models — so neither panel has empty bars.

1. **Total params bar chart** (log y) — see the 70× range from ViLT (117M) to DeepSeek-VL2-Small / Pixtral-12B.
2. **Stacked V/F/L/H bars** for the 18 standard-schema models — visualize allocation pattern.
3. **Stacked native bars** for Chameleon and Janus-Pro — uses the extended component palette.
4. **V/L ratio by family** — the family-dependent asymmetry headline. Native_fused models excluded (ratio undefined).
5. **Trainable vs total** — visualizes the freezing argument (F5). Schema-aware totals.

In [ ]:
if results:
    plot_df = params_df.copy().sort_values('total_M').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    schema_colors = {'standard_vflh': 'steelblue', 'native_fused': '#9B59B6', 'decoupled_vision': '#E74C3C'}
    bar_colors = [schema_colors.get(s, 'gray') for s in plot_df['schema']]
    bars = ax.barh(plot_df['name'], plot_df['total_M'], color=bar_colors, edgecolor='black')
    ax.set_xscale('log')
    ax.set_xlabel('Total parameters (M, log scale)')
    ax.set_title('Total params per model — Phase-2 n=20 roster (color = schema)')
    for bar, val in zip(bars, plot_df['total_M']):
        ax.annotate(f'{val:,.0f}M', xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
                    xytext=(5, 0), textcoords='offset points', va='center', fontsize=7)
    # Legend
    from matplotlib.patches import Patch
    legend_handles = [Patch(facecolor=c, label=s, edgecolor='black') for s, c in schema_colors.items()]
    ax.legend(handles=legend_handles, loc='lower right')
    ax.grid(True, which='both', axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_total_params_bar.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_total_params_bar.png"}')

In [ ]:
if results:
    # Stacked bar V/F/L/H per STANDARD-SCHEMA model, sorted by family then year
    sb_df = params_df[params_df['schema'] == 'standard_vflh'].copy().sort_values(['family', 'year', 'name']).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(sb_df))
    width = 0.7
    colors = {'vision': '#2ECC71', 'fusion': '#E67E22', 'language': '#3498DB', 'head': '#95A5A6'}
    bottoms = np.zeros(len(sb_df))
    for cname in ['vision', 'fusion', 'language', 'head']:
        col = f'{cname}_total'
        if col not in sb_df.columns:
            continue
        vals = (sb_df[col].fillna(0) / 1e6).values
        ax.bar(x, vals, width=width, bottom=bottoms, color=colors[cname], edgecolor='black', linewidth=0.5, label=cname)
        bottoms += vals
    ax.set_xticks(x)
    ax.set_xticklabels(sb_df['name'], rotation=60, ha='right', fontsize=9)
    ax.set_ylabel('Parameters (M, log scale)')
    ax.set_yscale('log')
    ax.set_title('Per-component parameter breakdown (STANDARD V/F/L/H schema, n={})'.format(len(sb_df)))
    ax.legend(loc='upper left', framealpha=0.9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_stacked_vflh_standard.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_stacked_vflh_standard.png"}')

In [ ]:
if results:
    # Stacked bar for NATIVE-SCHEMA models — chameleon + janus.
    native_df = params_df[params_df['schema'].isin(['native_fused', 'decoupled_vision'])].copy().sort_values(['schema', 'name']).reset_index(drop=True)
    if len(native_df) == 0:
        print('No native-schema models in the result set.')
    else:
        fig, ax = plt.subplots(figsize=(8, 6))
        x = np.arange(len(native_df))
        width = 0.6
        native_palette = {
            'fused_transformer': '#9B59B6',  # purple (native fused body)
            'vision_und':        '#1ABC9C',  # teal (Janus understanding pathway)
            'vision_gen':        '#16A085',  # darker teal (Janus generation pathway)
            'embeddings':        '#F39C12',  # orange (Chameleon image-token vocab)
            'fusion':            '#E67E22',  # standard fusion
            'language':          '#3498DB',  # standard language
            'head':              '#95A5A6',  # gray head
        }
        bottoms = np.zeros(len(native_df))
        for cname in ['fused_transformer', 'vision_und', 'vision_gen', 'embeddings', 'fusion', 'language', 'head']:
            col = f'{cname}_total'
            if col not in native_df.columns:
                continue
            vals = (native_df[col].fillna(0) / 1e6).values
            if vals.sum() == 0:
                continue
            ax.bar(x, vals, width=width, bottom=bottoms, color=native_palette.get(cname, 'gray'), edgecolor='black', linewidth=0.5, label=cname)
            bottoms += vals
        ax.set_xticks(x)
        ax.set_xticklabels([f'{n}\n({s})' for n, s in zip(native_df['name'], native_df['schema'])], fontsize=10)
        ax.set_ylabel('Parameters (M)')
        ax.set_title('Per-component breakdown — NATIVE-SCHEMA models\n(Chameleon: native fused; Janus-Pro: decoupled vision)')
        ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'nb02_stacked_native.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved to {FIG_DIR / "nb02_stacked_native.png"}')

In [ ]:
if results:
    # V/L ratio by family — exclude native_fused (where it's undefined)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    rdf = params_df.dropna(subset=['v_to_l_ratio']).copy()
    rdf = rdf[rdf['v_to_l_ratio'] > 0].sort_values('v_to_l_ratio')
    bars = axes[0].barh(rdf['name'], rdf['v_to_l_ratio'], color='coral', edgecolor='black')
    axes[0].axvline(1.0, color='black', linestyle='--', alpha=0.5, label='V = L (balanced)')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Vision / Language parameter ratio (log)')
    axes[0].set_title(f'V/L ratio per model (n={len(rdf)}; native_fused excluded — ratio undefined)')
    axes[0].legend()
    axes[0].grid(True, which='both', axis='x', linestyle='--', alpha=0.3)
    for bar, val in zip(bars, rdf['v_to_l_ratio']):
        axes[0].annotate(f'{val:.3f}', xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
                         xytext=(3, 0), textcoords='offset points', va='center', fontsize=7)

    # Right panel: vision% per family (uses schema-aware vision_pct)
    fam_summary = params_df.groupby('family').agg(
        n=('name', 'count'),
        avg_vision_pct=('vision_pct', 'mean'),
        avg_language_pct=('language_pct', 'mean'),
    ).sort_values('avg_vision_pct', ascending=False)
    axes[1].barh(fam_summary.index, fam_summary['avg_vision_pct'], color='#2ECC71', edgecolor='black', label='vision-like %')
    axes[1].barh(fam_summary.index, fam_summary['avg_language_pct'], left=fam_summary['avg_vision_pct'],
                 color='#3498DB', edgecolor='black', label='language-like %')
    axes[1].set_xlim(0, 105)
    axes[1].set_xlabel('Mean parameter share within family (%)')
    axes[1].set_title('Average V vs L share by architectural family (schema-aware)')
    axes[1].legend(loc='lower right')
    for i, (idx, row) in enumerate(fam_summary.iterrows()):
        axes[1].annotate(f'n={int(row["n"])}', xy=(101, i), va='center', fontsize=7)
    axes[1].grid(True, axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_vlratio_and_family.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_vlratio_and_family.png"}')

In [ ]:
if results:
    # Trainable vs total (schema-aware totals) — the freezing visualization (F5)
    sb_df = params_df.copy().sort_values(['family', 'year']).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(sb_df))
    width = 0.35
    ax.bar(x - width/2, sb_df['total_M'], width=width, color='lightgray', edgecolor='black', label='Total params (schema-aware)')
    ax.bar(x + width/2, sb_df['trainable_total_M'], width=width, color='#E74C3C', edgecolor='black', label='Trainable (canonical recipe)')
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(sb_df['name'], rotation=60, ha='right', fontsize=9)
    ax.set_ylabel('Parameters (M, log scale)')
    ax.set_title('Total vs trainable parameters — the Phase-2 F-5 fix in action\n'
                 'tall gray bars + short red bars = lots of frozen capacity (BLIP-2, IDEFICS, Idefics2, LLaVA family)')
    ax.legend(loc='upper left')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    for i, (_, row) in enumerate(sb_df.iterrows()):
        ax.annotate(f'{row["trainable_pct"]:.0f}%', xy=(i, max(row['trainable_total_M'], 0.01)),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7, color='#E74C3C')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_trainable_vs_total.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_trainable_vs_total.png"}')

---
## 10. Per-family summary statistics

Aggregate by family. This is one of the headline tables for the paper.

In [ ]:
if results:
    family_summary = params_df.groupby('family').agg(
        n_models       = ('name', 'count'),
        avg_total_M    = ('total_M', 'mean'),
        avg_vision_pct = ('vision_pct', 'mean'),
        avg_fusion_pct = ('fusion_pct', 'mean'),
        avg_language_pct = ('language_pct', 'mean'),
        avg_v_to_l_ratio = ('v_to_l_ratio', 'mean'),
        avg_trainable_pct = ('trainable_pct', 'mean'),
        models = ('name', lambda s: ', '.join(s)),
    ).round(1).sort_values('avg_vision_pct', ascending=False)
family_summary if results else None

In [ ]:
# Cross-family interpretation
if results:
    print('=== Family-level allocation patterns (schema-aware) ===\n')
    print('Vision-dominant families (mean vision-like % > 50%):')
    vd = family_summary[family_summary['avg_vision_pct'] > 50]
    for fam, row in vd.iterrows():
        print(f'  {fam:35s} mean vision-like = {row["avg_vision_pct"]:.1f}%  (n={int(row["n_models"])})')
    print('\nLanguage-dominant families (mean language-like % > 70%):')
    ld = family_summary[family_summary['avg_language_pct'] > 70]
    for fam, row in ld.iterrows():
        print(f'  {fam:35s} mean language-like = {row["avg_language_pct"]:.1f}%  (n={int(row["n_models"])})')
    print('\nBalanced / native families (neither dominates, or undefined):')
    bal = family_summary[(family_summary['avg_vision_pct'] <= 50) & (family_summary['avg_language_pct'] <= 70)]
    for fam, row in bal.iterrows():
        print(f'  {fam:35s} V={row["avg_vision_pct"]:.1f}%  L={row["avg_language_pct"]:.1f}%  (n={int(row["n_models"])})')

---
## 11. Failures (debugging)

If any model failed, the row appears here. Look at the corresponding log file in `../outputs/logs/` for the full traceback.

In [ ]:
if failures:
    failures_df = pd.DataFrame(failures)
    print(f'{len(failures)} failures:')
    failures_df
else:
    print('No failures — all models succeeded.')

---
## 12. Sanity checks against legacy Sunita-1/2/3 numbers

Cross-reference against the legacy notebooks. **Note:** the 2026-05-21 Colab run found a ~152M discrepancy on LLaVA-1.5 (Sunita-1: 7,063.43M, Phase-2: 6,911.11M). §12b investigates.

In [ ]:
LEGACY_CHECKS = [
    # (model name, component, expected param count, source, delta_pct_tolerance)
    ('CLIP-ViT-L/14',     'vision',     303_179_776,   'Sunita-1 cell 12', 0.0),
    ('CLIP-ViT-L/14',     'language',   123_060_480,   'Sunita-1 cell 12', 0.0),
    ('CLIP-ViT-L/14',     'fusion',       1_376_256,   'Sunita-1 cell 12', 0.0),
    ('LLaVA-1.5-7B',      'vision',     303_507_456,   'Sunita-1 cell 9',  0.0),
    ('LLaVA-1.5-7B',      'fusion',      20_979_712,   'Sunita-1 cell 9 (projector)',  0.0),
    ('LLaVA-1.5-7B',      'language',  6_738_939_904,  'Sunita-1 cell 9',  3.0),  # tolerance for weight-tying
]

if results:
    rows = []
    for name, comp, expected, source, tol_pct in LEGACY_CHECKS:
        match = params_df[params_df['name'] == name]
        if match.empty:
            rows.append({'model': name, 'component': comp, 'expected': expected, 'got': None,
                          'delta': None, 'pct_off': None, 'status': 'NOT RUN', 'source': source})
            continue
        col = f'{comp}_total'
        got = int(match.iloc[0].get(col, 0) or 0)
        delta = got - expected
        pct_off = 100 * abs(delta) / max(expected, 1)
        if delta == 0:
            status = '✓ exact'
        elif pct_off <= tol_pct:
            status = f'✓ within {tol_pct}% tolerance'
        else:
            status = f'⚠ OFF by {delta:+,} ({pct_off:.2f}%)'
        rows.append({'model': name, 'component': comp, 'expected': expected, 'got': got,
                      'delta': delta, 'pct_off': round(pct_off, 2), 'status': status, 'source': source})
    pd.DataFrame(rows)

### 12b. Investigation — LLaVA-1.5 ~152M discrepancy with Sunita-1

From the 2026-05-21 Runpod run:
* Sunita-1 cell 9 (legacy):  LLaVA-1.5 total = **7,063,427,072** (303.5M vision + 21.0M projector + 6,738.9M LLM, head=0)
* Phase-2 (2026-05-21):       LLaVA-1.5 total = **6,911,113,728** — 152M short

Most likely cause: HuggingFace `LlavaForConditionalGeneration` may now have `tie_word_embeddings=True` so the `lm_head` weights are shared with the input embedding table. The previous Sunita-1 count may have double-counted them. The next cell loads LLaVA-1.5 directly and checks.

In [ ]:
# Optional investigation. Requires ~16 GB GPU memory; skip if Pixtral / DeepSeek already loaded.
from transformers import LlavaForConditionalGeneration
llava_spec = next(m for m in MODELS if m.name == 'LLaVA-1.5-7B')
model = LlavaForConditionalGeneration.from_pretrained(
    llava_spec.hf_id, torch_dtype=torch.float16, low_cpu_mem_usage=True
)
tie = getattr(model.config, 'tie_word_embeddings', 'N/A')
print(f'tie_word_embeddings: {tie}')
all_params = list(model.parameters())
print(f'Total params (sum p.numel()): {sum(p.numel() for p in all_params):,}')
deduped = {id(p): p for p in all_params}
print(f'Total UNIQUE params (deduped by tied tensor id): {sum(p.numel() for p in deduped.values()):,}')
delta = sum(p.numel() for p in all_params) - sum(p.numel() for p in deduped.values())
print(f'Delta from tying: {delta:,} ({delta/1e6:.1f}M)')
lm_head_w = model.lm_head.weight
embed_w = model.model.language_model.embed_tokens.weight if hasattr(model.model.language_model, 'embed_tokens') else None
if embed_w is not None:
    print(f'lm_head.weight is embed_tokens.weight? {lm_head_w.data_ptr() == embed_w.data_ptr()}')
    print(f'lm_head.weight shape: {tuple(lm_head_w.shape)} = {lm_head_w.numel():,} params')
    print(f'If tied, Sunita-1 double-counted the head — Phase-2 dedupes via the same tensor pointer.')
del model; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## 13. Save the master parameter table to disk

The per-model JSONs are already saved (incrementally merged by `run_full_audit`). We additionally save the aggregated DataFrame as CSV + the family summary.

In [ ]:
if not results:
    print('Nothing to save — no successful results.')
else:
    csv_master = MEASUREMENT_DIR / 'nb02_parameter_summary.csv'
    csv_family = MEASUREMENT_DIR / 'nb02_family_summary.csv'
    csv_wide = MEASUREMENT_DIR / 'nb02_parameter_wide.csv'
    params_df.to_csv(csv_master, index=False)
    family_summary.to_csv(csv_family)
    wide_view.to_csv(csv_wide, index=False)
    print(f'Saved: {csv_master}')
    print(f'Saved: {csv_family}')
    print(f'Saved: {csv_wide}')
    print(f'Per-model JSONs: {MEASUREMENT_DIR}/<hf_id>.json')
    print(f'Verbose .txt logs: {LOG_DIR}/')
    print(f'Figures: {FIG_DIR}/nb02_*.png')
    print()
    print(f'Models with per-model JSONs ({len(list(MEASUREMENT_DIR.glob("*.json")))} files):')
    for j in sorted(MEASUREMENT_DIR.glob('*.json')):
        print(f'  {j.name}')

---
## Next steps

* If everything above looks right → continue to **`03_flops_analysis.ipynb`** for FLOPs measurements.
* If a model failed → open `outputs/logs/<timestamp>_02_parameter_analysis_v3_nb__<that_model>.txt`. Common causes:
  * Missing `HF_TOKEN` for gated models (LLaMA / IDEFICS / Pixtral / Llama-3.2-Vision)
  * `trust_remote_code` not allowed for InternVL3 / Molmo / DeepSeek-VL2 / Janus-Pro
  * OOM on Pixtral-12B → pass `quantization='int8'` in the run loop
  * Disk full → set `CLEANUP_HF_CACHE_BETWEEN_MODELS = True` at the top, restart kernel, re-run
* If a number looks wrong → use `01_inspect_a_single_model.ipynb` to load the model interactively and walk the module tree.
* If Chameleon or Janus look weird in §10 family summary — that's the schema-aware aggregation. Check the wide table in §8 to see the full breakdown.

*End of notebook 02 (v3 — 2026-05-22). Changes from v2: schema-aware aggregation, native-model handling, Runpod-only setup, extended wide view.*